In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np

def predict_bin_fill_time(file_path, target_level=100, recent_days=14):
    """
    Predicts the number of days until a bin reaches the target level (100%)
    based on historical filling rates.

    The function calculates two rates: a Global Average (stable) and a Recent
    Average (adaptive) to give a well-rounded prediction.

    Args:
        file_path (str): The path to the CSV dataset.
        target_level (int): The level (percentage) considered "full".
        recent_days (int): The number of recent days to use for the adaptive prediction.
    """
    print(f"--- Bin Fill Prediction Analysis (Target: {target_level}%) ---")

    # 1. Load and Preprocess Data
    try:
        df = pd.read_csv(file_path)

        df.columns = ['Date', 'Level']

        df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
        df = df.sort_values(by='Date').set_index('Date')


        df['Level'] = pd.to_numeric(df['Level'], errors='coerce')
        df = df.dropna()

    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return
    except Exception as e:
        print(f"An error occurred during data loading: {e}")
        return

    # 2. Calculate Daily Change
    # .diff() calculates the difference between the current row and the previous row
    df['Daily_Change'] = df['Level'].diff()

    # Get the latest reading
    current_level = df['Level'].iloc[-1]
    last_date = df.index[-1].strftime('%Y-%m-%d')

    print(f"Latest Reading Date: {last_date}")
    print(f"Current Bin Level: {current_level}%\n")

    remaining_capacity = target_level - current_level

    if remaining_capacity <= 0:
        print("Prediction: The bin is already at or above the target level (100%).")
        return

    # --- Prediction Method 1: Global Average Filling Rate (Most Stable) ---

    # 3. Calculate Global Average Filling Rate
    # Filter for positive changes only (only count days where the bin was filling)
    global_filling_changes = df[df['Daily_Change'] > 0]['Daily_Change']

    if global_filling_changes.empty:
        global_avg_daily_fill = 0
    else:
        # Use the mean of all positive daily changes
        global_avg_daily_fill = global_filling_changes.mean()

    # 4. Global Prediction
    if global_avg_daily_fill > 0:
        days_to_fill_global = remaining_capacity / global_avg_daily_fill
        prediction_days_global = round(days_to_fill_global)
        prediction_date_global = (df.index[-1] + pd.Timedelta(days=prediction_days_global)).strftime('%Y-%m-%d')

        print("--- Global Historical Trend (Averaging all history) ---")
        print(f"Average Daily Fill Rate: {global_avg_daily_fill:.2f} percentage points/day")
        print(f"Estimated Days to Full: {prediction_days_global} days")
        print(f"Predicted Date to be Full: {prediction_date_global}\n")
    else:
        print("Global Trend: Cannot calculate a reliable filling rate (no positive changes observed).")


    # 5. Calculate Recent Average Filling Rate
    recent_df = df.last(f'{recent_days}D')
    recent_filling_changes = recent_df[recent_df['Daily_Change'] > 0]['Daily_Change']

    if recent_filling_changes.empty or len(recent_df) < 5:
        print(f"Recent Trend: Not enough data points ({len(recent_df)}) or no filling trend observed in the last {recent_days} days.")
    else:
        recent_avg_daily_fill = recent_filling_changes.mean()

        # 6. Recent Prediction
        if recent_avg_daily_fill > 0:
            days_to_fill_recent = remaining_capacity / recent_avg_daily_fill
            prediction_days_recent = round(days_to_fill_recent)
            prediction_date_recent = (df.index[-1] + pd.Timedelta(days=prediction_days_recent)).strftime('%Y-%m-%d')

            print(f"--- Recent Trend (Last {recent_days} Days) ---")
            print(f"Average Daily Fill Rate: {recent_avg_daily_fill:.2f} percentage points/day")
            print(f"Estimated Days to Full: {prediction_days_recent} days")
            print(f"Predicted Date to be Full: {prediction_date_recent}\n")
        else:
            print(f"Recent Trend: Filling rate is zero in the last {recent_days} days.")

    print("--- End of Analysis ---")

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Hackathon/timeseries_bin_level .csv')
FILE_NAME = '/content/drive/MyDrive/Hackathon/timeseries_bin_level .csv'
predict_bin_fill_time(FILE_NAME)

--- Bin Fill Prediction Analysis (Target: 100%) ---
Latest Reading Date: 2024-04-30
Current Bin Level: 73%

--- Global Historical Trend (Averaging all history) ---
Average Daily Fill Rate: 17.54 percentage points/day
Estimated Days to Full: 2 days
Predicted Date to be Full: 2024-05-02

--- Recent Trend (Last 14 Days) ---
Average Daily Fill Rate: 9.75 percentage points/day
Estimated Days to Full: 3 days
Predicted Date to be Full: 2024-05-03

--- End of Analysis ---


/tmp/ipython-input-2061649530.py:84: FutureWarning: last is deprecated and will be removed in a future version. Please create a mask and filter using `.loc` instead
  recent_df = df.last(f'{recent_days}D')
